# BHoM classes in Databricks: from C# object model to Delta tables

`bhom-schema` is a Python package whose classes are **generated from the BHoM C# object model (oM)**. Every class keeps the names, types, units and descriptions of its C# original, checks every value you give it, and reads and writes the same JSON as BHoM. This notebook shows how those classes meet Databricks: you create Delta tables from the classes, write Python objects into them, and read the rows back as Python objects.

The classes used here are the real ones in `bhom_schema.DataSchema_oM` (the precedent building data: `Project`, `Building`, `Storey`, `Grid`, `Beam`, `Column`, and the demo class `ExampleObject`).

**What you will see**

1. Install or update the package
2. Settings (where the demo tables go)
3. Import the classes and use them as ordinary Python objects
4. How a type becomes a column type
5. A type with no column type, the error, and the fix (the `Any` case)
6. Generate the SQL and create tables for several classes
7. Write instances of the classes to the tables
8. Read table rows back as Python objects
9. A class that has properties of another class
10. Delta refuses a wrong enum value
11. Clean up

**To run it:** attach a cluster or serverless compute with Python 3.12 or later, set the catalog and schema in section 2 (you need permission to create tables there), and use *Run all*. Nothing in this notebook is expected to fail. It only creates tables whose names start with `bhom_demo_`, and section 11 can drop them.

## 1. Install or update the package

The package is installed per notebook with `%pip`. `--upgrade --pre` fetches the latest build even if one is already installed, and `--no-cache-dir` avoids a stale download. The Python process is restarted afterwards so the new version is the one that is imported. `%pip` and the restart must each be in a cell of their own.

In [ ]:
%pip install --upgrade --pre --no-cache-dir --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ bhom-schema

In [ ]:
dbutils.library.restartPython()

In [ ]:
import bhom_schema

print("package version :", bhom_schema.__version__)
print("BHoM version    :", bhom_schema.__bhom_version__)

## 2. Settings

The demo tables are created in `catalog.schema` and are named `bhom_demo_<class>`. **Change the two widgets at the top of the notebook to a catalog and schema of your own** (the schema must already exist and you must be allowed to create tables in it; the defaults are only an example).

In [ ]:
dbutils.widgets.text("catalog", "student_area", "Catalog")
dbutils.widgets.text("schema", "vcl", "Schema")
dbutils.widgets.dropdown("cleanup", "no", ["no", "yes"], "Drop the demo tables at the end")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

def table_name(suffix):
    """Returns the full name of a demo table: catalog.schema.bhom_demo_<suffix>."""
    return f"{CATALOG}.{SCHEMA}.bhom_demo_{suffix}"

def show_sql(statement, width=130):
    """Prints a SQL statement, cutting the long column comments so that it fits on the screen."""
    for line in statement.splitlines():
        print(line if len(line) <= width else line[:width] + " ...")

# Fails here, with a clear message, when the schema does not exist or is not readable
existing = spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{SCHEMA}`").count()
print(f"tables will be created in {CATALOG}.{SCHEMA} ({existing} tables there already)")

## 3. Import the classes and use them

Each C# class is a Python module with a class of the same name, in the same folder structure as the C# namespace. A class is a normal Python object: create it with keyword arguments named like the C# properties, read and set its fields, and every value is checked (see the validation notebook for much more).

The first cell imports what the rest of the notebook needs. **Expect:** no output.

In [ ]:
from pydantic import ValidationError

from bhom_schema.DataSchema_oM.Project import Project
from bhom_schema.DataSchema_oM.Building import Building
from bhom_schema.DataSchema_oM.Storey import Storey
from bhom_schema.DataSchema_oM.Grid import Grid
from bhom_schema.DataSchema_oM.Elements.Beam import Beam
from bhom_schema.DataSchema_oM.Elements.Column import Column
from bhom_schema.DataSchema_oM.Enums.StructuralMaterial import StructuralMaterial
from bhom_schema.DataSchema_oM.ExampleObject import ExampleObject
from bhom_schema.DataSchema_oM.ExampleStatus import ExampleStatus
from bhom_schema.DataSchema_oM.ExampleFinish import ExampleFinish

A class carries the documentation of its C# original. **Expect:** the summary of `Building`, then each of its fields with the description from C#.

In [ ]:
print(Building.__doc__)
print()
for name, field in Building.model_fields.items():
    print(f"{name:28} {(field.description or '')[:70]}")

Now some precedent buildings. Enums are real Python enums (`StructuralMaterial.SteelHotRolled`), a member can also be given by its C# name as text, and a field the C# class has but you leave out takes the C# default. **Expect:** no output.

In [ ]:
PROJECTS = [
    Project(Name="Riverside Tower", Number="BH-1001"),
    Project(Name="Harbour Library", Number="BH-1002"),
    Project(Name="Kings Cross Lab", Number="BH-1003"),
]

BUILDINGS = [
    Building(
        Name="Riverside Tower", StoreysAboveGround=24, StoreysBelowGround=2,
        MaterialVolumesAboveGround={"Concrete": 9800.0, "Steel": 640.5}, MaterialVolumesBelowGround={"Concrete": 4100.0}),
    Building(
        Name="Harbour Library", StoreysAboveGround=5, StoreysBelowGround=1,
        MaterialVolumesAboveGround={"Timber": 1250.0, "Concrete": 900.0}, MaterialVolumesBelowGround={"Concrete": 700.0}),
    # No basement and no volumes below ground: those fields stay empty (None)
    Building(Name="Kings Cross Lab", StoreysAboveGround=9, StoreysBelowGround=0, MaterialVolumesAboveGround={"Steel": 1320.0, "Concrete": 2100.0}),
]

STOREYS = [
    Storey(Name="Riverside typical", TypicalHeight=3.9, NonRoofArea=1850.0, RoofArea=1620.0),
    Storey(Name="Harbour typical", TypicalHeight=4.2, NonRoofArea=960.0, RoofArea=900.0),
    Storey(Name="Kings Cross typical", TypicalHeight=4.5, NonRoofArea=1210.0, RoofArea=1100.0),
]

GRIDS = [
    Grid(Name="Riverside grid", TypicalXSpacing=9.0, TypicalYSpacing=9.0),
    Grid(Name="Harbour grid", TypicalXSpacing=7.5, TypicalYSpacing=12.0),
    Grid(Name="Kings Cross grid", TypicalXSpacing=8.4, TypicalYSpacing=8.4),
]

BEAMS = [
    Beam(
        Name="Riverside beams", TypicalMaterial=StructuralMaterial.SteelHotRolled, TypicalSize="UB 457x191x67", TypicalCountPerSlab=48,
        IsRefurbished=False, MaterialDistribution={StructuralMaterial.SteelHotRolled: 1020.0, StructuralMaterial.Composite: 96.0}),
    Beam(
        Name="Harbour beams", TypicalMaterial=StructuralMaterial.TimberGlulam, TypicalSize="GL28h 200x500", TypicalCountPerSlab=30,
        IsRefurbished=False, MaterialDistribution={StructuralMaterial.TimberGlulam: 210.0}),
    # The member can be given by its C# name as text
    Beam(Name="Kings Cross beams", TypicalMaterial="SteelHotRolled", TypicalSize="UB 305x165x40", TypicalCountPerSlab=36, IsRefurbished=True),
]

COLUMNS = [
    Column(Name="Riverside columns", TypicalMaterial=StructuralMaterial.ConcreteInSitu, TypicalSize="600 x 600", TypicalCountPerSlab=40),
    Column(Name="Harbour columns", TypicalMaterial=StructuralMaterial.TimberGlulam, TypicalSize="320 x 320", TypicalCountPerSlab=24),
]

Read the values back as plain Python. **Expect:** the enum member, the dictionary with enum keys, and the start of the JSON that BHoM would write (`_t` is the C# type name).

In [ ]:
beam = BEAMS[0]
print(beam.TypicalMaterial)
print(beam.MaterialDistribution)
print(beam.model_dump_json()[:150], "...")

A wrong value is refused at once, with a message that names the field. **Expect:** two lines, the first about `TypicalHeight` and the second listing the members of `StructuralMaterial`.

In [ ]:
for mistake in (lambda: Storey(TypicalHeight="tall"), lambda: Beam(TypicalMaterial="Unobtainium")):
    try:
        mistake()
    except ValidationError as error:
        first = error.errors()[0]
        print(f"{first['loc'][0]}: {first['msg'][:110]}")

## 4. How a type becomes a column type

`bhom_schema.tables` turns a class into a table, one class at a time. A field whose type is a **primitive** (a number, text, boolean, date, enum, or a list or dictionary of those) gets its column automatically. `column_type` shows the Spark SQL type for a Python type, and `describe` lists the columns that a class would have, key first. **Expect:** `DOUBLE`, `ARRAY<STRING>`, `MAP<STRING, DOUBLE>`, `DECIMAL(38,10)`, then the columns of `Storey`; the last two lines are fields that have no column yet.

In [ ]:
from decimal import Decimal
from typing import Any

from bhom_schema.tables import (
    BASE_SKIPPED, Json, Ref, Skip, Struct, UnmappedType, column_type, create_table_sql, describe,
    enum_constraints_sql, rows_to_objects)
from bhom_schema.tables.spark import create_table, fetch_by_ids, read_objects, read_rows, write_objects
from bhom_schema.tables.sql import quote_table

print(column_type(float))
print(column_type(list[str]))
print(column_type(dict[str, float]))
print(column_type(Decimal))
print()
for column in describe(Storey):
    print(f"{column.name:16} {str(column.type):16} {column.decision or 'needs a decision'}")

## 5. See the error, then fix it: the `Any` type

The last two fields of `Storey`, `Fragments` and `CustomData`, are on **every** BHoM object. `CustomData` is `dict[str, Any]` in Python, a dictionary that can hold anything, and `Any` has no SQL column type. The helpers never guess: they refuse to build SQL until you decide what to do with each such field. **Expect:** two messages. The first says that `Any` has no column type, the second lists the two undecided fields of `Storey` and the choices for each.

In [ ]:
try:
    column_type(Any)
except UnmappedType as error:
    print(error)

print()
try:
    create_table_sql(Storey, table_name("storey"))
except UnmappedType as error:
    print(error)

**The fix** is a decision for each of those fields, passed as `fields=`:

| Decision | The field is stored as |
| --- | --- |
| `Skip()` | nothing: no column, and a row read back keeps the default of the field |
| `Json()` | the JSON that the class writes for it, in a `STRING` column |
| `Struct()` | a `STRUCT` column (or an `ARRAY` of them for a list) that holds the fields of the other class |
| `Ref("table")` | the id of a row of another table, in a `<Field>_id` column |

`BASE_SKIPPED` is a ready-made `{"Fragments": Skip(), "CustomData": Skip()}` that almost every BHoM table wants. **Expect:** the `CREATE TABLE` statement for `Storey`, with the description of each C# property as a column comment.

In [ ]:
show_sql(create_table_sql(Storey, table_name("storey"), fields=BASE_SKIPPED))

`Beam` has another field with no column: `MaterialDistribution` is a dictionary whose **key is an enum** (`dict[StructuralMaterial, float]`), and a column of type `MAP` needs a text key. BHoM itself writes such a dictionary as a list of key/value objects, and `Json()` stores exactly that. **Expect:** a message that names `MaterialDistribution` as the one field that is left (the base fields are decided), and then the statement works.

In [ ]:
try:
    create_table_sql(Beam, table_name("beam"), fields=BASE_SKIPPED)
except UnmappedType as error:
    print(error)

BEAM_FIELDS = {**BASE_SKIPPED, "MaterialDistribution": Json()}
print()
show_sql(create_table_sql(Beam, table_name("beam"), fields=BEAM_FIELDS))

## 6. Generate the SQL and create the tables

`create_table` runs the statement that `create_table_sql` writes, with `CREATE OR REPLACE TABLE` so the notebook can be run again. Each table is a Delta table with the class description as table comment, and the BHoM type and package version as table properties. **Expect:** one `created` line for each of the six tables, then the columns of `bhom_demo_building`.

In [ ]:
TABLES = {
    "project": (Project, BASE_SKIPPED),
    "building": (Building, BASE_SKIPPED),
    "storey": (Storey, BASE_SKIPPED),
    "grid": (Grid, BASE_SKIPPED),
    "beam": (Beam, BEAM_FIELDS),
    "column": (Column, BEAM_FIELDS),
}

OBJECTS = {"project": PROJECTS, "building": BUILDINGS, "storey": STOREYS, "grid": GRIDS, "beam": BEAMS, "column": COLUMNS}

for suffix, (model, fields) in TABLES.items():
    create_table(spark, model, table_name(suffix), fields=fields, replace=True)
    print("created", table_name(suffix))

display(spark.sql(f"DESCRIBE TABLE {quote_table(table_name('building'))}"))

## 7. Write class instances to the tables

`write_objects` appends a list of objects of one class to the table of that class, and returns the key of each row in order. For a BHoM object the key is its `BHoM_Guid`. **Expect:** the number of rows written to each table, then the rows of the building table.

In [ ]:
KEYS = {}
for suffix, (model, fields) in TABLES.items():
    KEYS[suffix] = write_objects(spark, OBJECTS[suffix], table_name(suffix), fields=fields)
    print(f"{suffix:9} wrote {len(KEYS[suffix])} rows")

display(spark.table(quote_table(table_name("building"))))

## 8. Read the rows back as Python objects

`read_rows` returns the rows as plain dictionaries (before any validation), and `read_objects` returns objects of the class. Every row is validated as it becomes an object, so a table that someone edited by hand cannot hand you a wrong value. `keyed=True` returns a dictionary by key, which is how the rows are matched to the objects that were written. **Expect:** one raw row (the enum is its text name, the `MaterialDistribution` the JSON text), then `True` for every table: what was read is equal to what was written.

In [ ]:
print(read_rows(spark, table_name("beam"), limit=1)[0])
print()
for suffix, (model, fields) in TABLES.items():
    found = read_objects(spark, model, table_name(suffix), fields=fields, keyed=True)
    matches = all(found[key] == original for key, original in zip(KEYS[suffix], OBJECTS[suffix]))
    print(f"{suffix:9} {len(found)} objects read, equal to the ones written: {matches}")

A SQL condition picks the rows, and only those become objects. **Expect:** one beam, with its enum members and its dictionary rebuilt (the keys are `StructuralMaterial` members again, not text).

In [ ]:
large_beams = read_objects(spark, Beam, table_name("beam"), fields=BEAM_FIELDS, where="`TypicalCountPerSlab` > 40")
print(len(large_beams), large_beams[0].Name, large_beams[0].TypicalMaterial)
print(large_beams[0].MaterialDistribution)

If some rows are wrong, `errors="collect"` reads the good ones and lists the bad ones with the position, the key and the message, instead of stopping at the first. **Expect:** `1 objects, 1 problem`, then the position and the message of the bad row.

In [ ]:
from uuid import uuid4

rows = [
    {"BHoM_Guid": str(uuid4()), "TypicalHeight": 3.5},
    {"BHoM_Guid": str(uuid4()), "TypicalHeight": "tall"},
]
objects, problems = rows_to_objects(Storey, rows, fields=BASE_SKIPPED, errors="collect")

print(f"{len(objects)} objects, {len(problems)} problem")
print(problems[0].index, problems[0].message.splitlines()[:3])

## 9. A class with properties of another class

Until now every field was a primitive or an enum. `ExampleObject` has fields that hold **other classes** (and other shapes with no column). `describe` lists them, with their types, as the fields that still need a decision. **Expect:** the list below: `MainGrid` is one `Grid`, `Storeys` a list of `Storey`, `Anything` any object that implements `IObject`, plus a tuple, an enum-keyed dictionary and `Any`.

In [ ]:
for column in describe(ExampleObject):
    for undecided in column.undecided_fields:
        print(f"{undecided.path:16} {undecided.type}")

The same rule applies as in section 5: nothing is guessed, and the error lists what is undecided with the choices. **Expect:** the message, ending with the four decisions.

In [ ]:
try:
    create_table_sql(ExampleObject, table_name("example"))
except UnmappedType as error:
    print(error)

Now decide each field. The choice is yours, and it is the same for every field of another class:

- **`MainGrid`** (one `Grid`) becomes a `Ref` to the `bhom_demo_grid` table of section 6: the example table holds only the id (`MainGrid_id`), and the grid is a row of its own, shared by anyone who points at it.
- **`Storeys`** (a list of `Storey`) becomes a `Struct`: an `ARRAY` of `STRUCT` held inside the row, so the storeys live and die with the example. The `Struct` has decisions too, for the fields of `Storey` (`BASE_SKIPPED` again).
- **`Anything`, `TotalsByStatus`, `Pair`** become `Json`: text, with the type recorded in it (`Anything` carries the `_t` of whatever class it holds, so it is rebuilt as the right class).
- **`Extra`** (`Any`) is skipped. It is not stored.

**Expect:** the columns that needed a decision, with the column type each decision gives.

In [ ]:
EXAMPLE_TABLE = table_name("example")
EXAMPLE_FIELDS = {
    **BASE_SKIPPED,
    "MainGrid": Ref(table_name("grid")),
    "Storeys": Struct(fields=BASE_SKIPPED),
    "Anything": Json(),
    "TotalsByStatus": Json(),
    "Pair": Json(),
    "Extra": Skip(),
}

for column in describe(ExampleObject, EXAMPLE_FIELDS):
    if column.decision != "primitive":
        print(f"{column.name:16} {column.decision:8} {str(column.type)[:80]}")

Create the table and write two examples. The example has a `Ref`, so the id of the grid row goes with it: `ids` has one entry per object, naming the key of the row that each `Ref` field points at. The keys come from section 7, where the grids were written. **Expect:** the table is created and two rows are written.

In [ ]:
EXAMPLES = [
    ExampleObject(
        Name="Riverside example", MainGrid=GRIDS[0], Storeys=[STOREYS[0], STOREYS[1]], Anything=BEAMS[0],
        TotalsByStatus={ExampleStatus.Draft: 2.5, ExampleStatus.Approved: 1.0}, Pair=(7, "seven"),
        Status=ExampleStatus.Approved, Finish=ExampleFinish.Painted | ExampleFinish.Insulated,
        Material=StructuralMaterial.ConcretePrecast, Labels=["precedent"], UniqueNames={"steel", "concrete"}, Extra={"anything": [1, 2, 3]}),
    ExampleObject(Name="Harbour example", MainGrid=GRIDS[1], Storeys=[STOREYS[2]], Anything=STOREYS[2]),
]

create_table(spark, ExampleObject, EXAMPLE_TABLE, fields=EXAMPLE_FIELDS, replace=True)
EXAMPLE_KEYS = write_objects(
    spark, EXAMPLES, EXAMPLE_TABLE, fields=EXAMPLE_FIELDS, ids=[{"MainGrid": KEYS["grid"][0]}, {"MainGrid": KEYS["grid"][1]}])
print(f"wrote {len(EXAMPLE_KEYS)} rows to {EXAMPLE_TABLE}")

In the raw row the grid is only an id, and the storeys are a list of structs. **Expect:** the id of the grid of the first example, and the two storeys with their heights.

In [ ]:
raw = {row["BHoM_Guid"]: row for row in read_rows(spark, EXAMPLE_TABLE)}
first_row = raw[EXAMPLE_KEYS[0]]

print("MainGrid_id :", first_row["MainGrid_id"])
print("Storeys     :", [storey["TypicalHeight"] for storey in first_row["Storeys"]])

Reading the objects back: a `Struct` is rebuilt without any help, since the data is in the row. A `Ref` needs the object it points at, which is in **another table**, so `read_objects` takes it as `held` (a dictionary of the objects by id, for each `Ref` field). Without it the field keeps its default and a warning says so. **Expect:** the warning text, then `MainGrid` is `None`.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    without_held = read_objects(spark, ExampleObject, EXAMPLE_TABLE, fields=EXAMPLE_FIELDS, keyed=True)

print(str(caught[0].message)[:120], "...")
print("MainGrid without held:", without_held[EXAMPLE_KEYS[0]].MainGrid)

To give `held`, fetch the rows of the grid table that the examples point at (`fetch_by_ids` reads only those), turn them into objects by id, and pass them. **Expect:** the grid is a real `Grid` with its spacing, the storeys are `Storey` objects, `Anything` is a `Beam` (rebuilt from the `_t` in the JSON), `Extra` is `None` because it is skipped, and `True` twice: everything else equals what was written.

In [ ]:
grid_ids = {row["MainGrid_id"] for row in raw.values()}
grid_rows = fetch_by_ids(spark, table_name("grid"), grid_ids, key="BHoM_Guid")
grids_by_id = rows_to_objects(Grid, grid_rows, fields=BASE_SKIPPED, keyed=True)

found = read_objects(spark, ExampleObject, EXAMPLE_TABLE, fields=EXAMPLE_FIELDS, held={"MainGrid": grids_by_id}, keyed=True)
example = found[EXAMPLE_KEYS[0]]

print("MainGrid      :", type(example.MainGrid).__name__, example.MainGrid.TypicalXSpacing)
print("Storeys       :", [type(storey).__name__ for storey in example.Storeys], [storey.TypicalHeight for storey in example.Storeys])
print("Anything      :", type(example.Anything).__name__, example.Anything.Name)
print("TotalsByStatus:", example.TotalsByStatus)
print("Finish        :", example.Finish, "(a flags value: Painted + Insulated = 1 + 2)")
print("Extra         :", example.Extra, "(was", EXAMPLES[0].Extra, "when written)")

# Extra is the only field that is not stored, so it is the only one that differs
print(found[EXAMPLE_KEYS[0]].model_dump(exclude={"Extra"}) == EXAMPLES[0].model_dump(exclude={"Extra"}))
print(found[EXAMPLE_KEYS[1]] == EXAMPLES[1])

## 10. Delta refuses a wrong enum value

Python refuses a wrong enum value at once, but a table can also be written by SQL, by a pipeline or by hand. `enum_constraints_sql` writes one `CHECK` constraint for each enum column of a class, so **Delta itself refuses a row** whose value is not a member (a `NULL` is allowed, and a flags enum takes names joined by `, `). **Expect:** the Python refusal with the members, the constraints that are added to the beam table, `accepted` for the good row, and `refused by Delta` for the bad one.

In [ ]:
BEAMS_TABLE = table_name("beam")

try:
    Beam(TypicalMaterial="Unobtainium")
except ValidationError as error:
    print("Python  :", error.errors()[0]["msg"][:100], "...")

for statement in enum_constraints_sql(Beam, BEAMS_TABLE, BEAM_FIELDS):
    print("Delta   :", statement[:110], "...")
    spark.sql(statement)

print()
for label, material in (("a good row", "TimberCLT"), ("a bad row", "Unobtainium")):
    insert_statement = f"INSERT INTO {quote_table(BEAMS_TABLE)} (`BHoM_Guid`, `TypicalMaterial`) VALUES ('{uuid4()}', '{material}')"
    try:
        spark.sql(insert_statement)
        print(f"accepted           {label}: TypicalMaterial = {material}")
    except Exception as error:
        print(f"refused by Delta   {label}: TypicalMaterial = {material}")
        print("   ", str(error).splitlines()[0][:150])

## 11. Clean up

Set the `cleanup` widget to `yes` (and run this cell) to drop every table this notebook created. The default is `no`, so you can look at the tables in the catalog explorer first.

In [ ]:
names = [table_name(suffix) for suffix in TABLES] + [EXAMPLE_TABLE]

if dbutils.widgets.get("cleanup") == "yes":
    for name in names:
        spark.sql(f"DROP TABLE IF EXISTS {quote_table(name)}")
        print("dropped", name)
else:
    print("kept these tables (set the cleanup widget to yes to drop them):")
    for name in names:
        print("  ", name)

## Summary

You installed the package, used real BHoM classes as Python objects, turned them into Delta tables, wrote and read them back, stored a class that holds other classes (`Ref`, `Struct`, `Json`, `Skip`), and saw type checks at three places: the Python object, the rows that are read, and the table itself. The notebook `Type-Validation-Showcase.ipynb` shows the first of those in detail, including what the editor shows while you type.